# Phase 9 — Spatial Intelligence
## Delhi NCR Urban Change Intelligence (2022 -> 2026)

This notebook executes the spatial intelligence pipeline across all usable patches (~1008 patches covering the full study area bounding box):
1. **Full-Scene Inference**: Runs the locked `BaselineChangeCNN` (with MC Dropout, N=20 passes) and rule-based change characterization.
2. **Real-World Area Metrics**: Converts 10m Sentinel-2 pixel counts to physical units (km², ha, %).
3. **Adaptive Baseline Urban Core Cutoff**: Derives data-driven T1 NDBI threshold and exports visual verification chips.
4. **Spatial Hotspots**: Detects statistically elevated built-up growth patches ($z \ge 1.645$) and multi-patch DBSCAN clusters.
5. **Proximity Analysis**: Computes exact Euclidean distance transform to the baseline urban core.
6. **Data Limitations**: Road vector proximity and sub-region administrative breakdowns are explicitly skipped due to absence of external vector datasets, strictly adhering to non-political geospatial framing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
REPO = '/content/delhi-ncr-satellite-change'
if not os.path.exists(REPO):
    !git clone https://github.com/karan02566-prog/delhi-ncr-satellite-change.git {REPO}
else:
    !cd {REPO} && git pull
if REPO not in sys.path:
    sys.path.append(REPO)


In [ ]:
import json
import torch
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

from src.models.baseline import BaselineChangeCNN
from src.data.label_generation import compute_ndbi, compute_valid_mask
from src.spatial.hotspots import (
    calculate_pixel_area,
    derive_adaptive_urban_cutoff,
    export_urban_cutoff_verification_chips,
    compute_patch_geometries,
    detect_spatial_hotspots,
    compute_urban_edge_proximity,
)
from src.spatial.spatial_analysis import (
    run_spatial_pipeline,
    export_spatial_hotspot_maps,
)

DRIVE_DIR = '/content/drive/MyDrive/delhi_ncr_change_detection'
T1_PATH = f'{DRIVE_DIR}/delhi_ncr_t1_2022_masked.tif'
T2_PATH = f'{DRIVE_DIR}/delhi_ncr_t2_2026_masked.tif'
MANIFEST_PATH = f'{DRIVE_DIR}/data_labels_split_manifest.json'
CHECKPOINT_PATH = f'{DRIVE_DIR}/checkpoints/baseline_best.pt'
OUTPUT_DIR = f'{DRIVE_DIR}/reports/spatial'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using compute device:', device)

# Load rasters (cast to float32 to manage RAM)
with rasterio.open(T1_PATH) as src_t1:
    t1_array = src_t1.read().astype(np.float32)
    raster_transform = src_t1.transform
    raster_crs = src_t1.crs

with rasterio.open(T2_PATH) as src_t2:
    t2_array = src_t2.read().astype(np.float32)

print('T1 shape:', t1_array.shape, 'T2 shape:', t2_array.shape)

# Load locked BaselineChangeCNN model (exact signature: in_channels=12, dropout_p=0.3)
model_base = BaselineChangeCNN(in_channels=12, dropout_p=0.3)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model_base.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
model_base.to(device)
print('Loaded locked BaselineChangeCNN checkpoint successfully.')


In [ ]:
# Load all usable patches across all splits (train, val, test, dropped_buffer)
with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

all_patch_records = []
for split_name, patch_list in manifest['splits'].items():
    for p in patch_list:
        rec = dict(p)
        rec['split'] = split_name
        all_patch_records.append(rec)

print(f'Total usable patches across full scene: {len(all_patch_records)}')
split_counts = pd.Series([p['split'] for p in all_patch_records]).value_counts().to_dict()
print('Split breakdown:', split_counts)


In [ ]:
# Compute adaptive baseline urban cutoff from T1 NDBI
t1_ndbi = compute_ndbi(t1_array)
valid_mask = compute_valid_mask(t1_array, t2_array)

urban_cutoff, urban_meta = derive_adaptive_urban_cutoff(t1_ndbi, valid_mask, std_mult=1.0)
print('Adaptive Baseline Urban Cutoff Metrics:')
for k, v in urban_meta.items():
    print(f'  {k}: {v}')

# Export visual verification chips (stratified sample) for manual review
urban_verif_dir = f'{OUTPUT_DIR}/urban_core_verification'
verif_chips = export_urban_cutoff_verification_chips(
    t1_array=t1_array,
    t1_ndbi=t1_ndbi,
    valid_mask=valid_mask,
    urban_cutoff=urban_cutoff,
    out_dir=urban_verif_dir,
    patch_size=256,
    n_samples=8,
)
print(f'Exported {len(verif_chips)} urban core verification chips to {urban_verif_dir}')


In [ ]:
# Execute full-scene spatial pipeline across all usable patches
patch_df, full_summary, test_summary = run_spatial_pipeline(
    model=model_base,
    patch_records=all_patch_records,
    t1_array=t1_array,
    t2_array=t2_array,
    device=device,
    out_dir=OUTPUT_DIR,
    n_passes=20,
    resume=True,
    transform=raster_transform,
    patch_size=256,
)

print('\n=== Full-Scene Area Summary ===')
print(f"Total Valid Area: {full_summary['total_valid_km2']:.2f} km² ({full_summary['total_valid_ha']:.1f} ha)")
print(f"Total Changed Area: {full_summary['total_changed_km2']:.2f} km² ({full_summary['overall_change_percentage']:.2f}% of valid area)")

print('\n=== Category Breakdown (Full Scene) ===')
for cat_name, cat_data in full_summary['categories'].items():
    print(f"{cat_name:>20}: {cat_data['area_km2']:>8.2f} km² ({cat_data['pct_of_valid_area']:>5.2f}% of valid | {cat_data['pct_of_total_change']:>5.2f}% of change)")

print('\n=== Test-Split Sanity Check (n=56) ===')
print(f"Test Set Change Rate: {test_summary['test_change_rate']*100:.2f}%")
print(f"Test Set Built-up Rate: {test_summary['test_builtup_rate']*100:.2f}%")
print(f"Mean Uncertainty (Change Pixels): {test_summary['mean_uncertainty_change']:.4f}")


In [ ]:
# Georeference patches and run spatial hotspot detection (DBSCAN)
patch_gdf = compute_patch_geometries(patch_df.to_dict(orient='records'), transform=raster_transform, patch_size=256)

candidate_gdf, clusters_gdf, hotspot_summary = detect_spatial_hotspots(
    patch_gdf=patch_gdf,
    z_threshold=1.645,
    patch_stride_km=2.56,
    min_samples=2,
)

print('\n=== Spatial Hotspot Detection Summary ===')
for k, v in hotspot_summary.items():
    print(f'  {k}: {v}')

if len(clusters_gdf) > 0:
    print('\nIdentified Multi-Patch Growth Clusters:')
    display_cols = ['cluster_id', 'patch_count', 'mean_z_score', 'total_builtup_km2']
    print(clusters_gdf[display_cols].to_string(index=False))


In [ ]:
# Compute proximity of newly expanded built-up area to baseline urban core
# Construct full-scene built-up expansion binary mask from characterized patches
H, W = valid_mask.shape
full_builtup_mask = np.zeros((H, W), dtype=bool)

for _, row_item in patch_df.iterrows():
    r, c = int(row_item['row']), int(row_item['col'])
    # If patch had built-up pixels, populate mask
    if row_item['builtup_pixels'] > 0:
        t1p = t1_array[:, r:r+256, c:c+256]
        t2p = t2_array[:, r:r+256, c:c+256]
        vp = valid_mask[r:r+256, c:c+256]
        d_ndbi = compute_ndbi(t2p) - compute_ndbi(t1p)
        full_builtup_mask[r:r+256, c:c+256] |= (d_ndbi > 0.05) & vp

proximity_results = compute_urban_edge_proximity(
    t1_ndbi=t1_ndbi,
    builtup_change_mask=full_builtup_mask,
    urban_cutoff=urban_cutoff,
    valid_mask=valid_mask,
    pixel_res_m=10.0,
)

print('\n=== Proximity to Baseline Urban Core ===')
print(f"Mean Distance to Existing Urban Core: {proximity_results['mean_distance_m']:.1f} meters")
print(f"Median Distance: {proximity_results['median_distance_m']:.1f} meters")
print(f"Within 500m (Direct Edge Expansion): {proximity_results['fraction_within_500m']*100:.1f}%")
print(f"Within 1000m (Near Urban Fringe): {proximity_results['fraction_within_1000m']*100:.1f}%")
print(f"Beyond 2000m (Isolated / Leapfrog Development): {proximity_results['fraction_beyond_2000m']*100:.1f}%")

with open(f'{OUTPUT_DIR}/urban_proximity_summary.json', 'w') as f:
    json.dump(proximity_results, f, indent=2)


In [ ]:
# Export spatial maps (density, uncertainty, hotspot clusters)
export_spatial_hotspot_maps(
    patch_df=patch_df,
    clusters_gdf=clusters_gdf,
    out_dir=OUTPUT_DIR,
    transform=raster_transform,
    patch_size=256,
)
print('All Phase 9 spatial intelligence artifacts successfully exported.')
